# Signup Abuse — Entity Linkage

Link accounts that plausibly share one real-world actor. **Precision-first**: never harm a legit customer.
Weights are **computed from the data** (Fellegi–Sunter `log2(m/u)`) — no hand-tuned numbers.

## Pipeline at a glance
```
 15,008 accounts ─► 1 NORMALIZE (all kept) ─► 2 GROUP shared keys ─► 3 ADDRESS fuzzy
        ─► 4 COMPUTE weights BASE=log2(m/u) ─► 5 WEIGHT+GATE edges ─► UNION-FIND
        ─► 251 clusters / 587 linked ─► clusters.csv · edges.csv · findings.md
```
Simple idea: turn every field into a key, let matching keys vote with a **computed** weight, keep the pairs whose votes clear a bar, and group them.

In [1]:
import re, ipaddress, math, csv
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

INPUT = Path("input"); OUT = Path("output"); OUT.mkdir(exist_ok=True)
df = pd.read_csv(INPUT / "signups.csv", dtype=str, keep_default_na=False)
N = len(df)
assert df["account_id"].is_unique
_DFI = df.set_index("account_id")
def _trunc(s, n=40):
    s = str(s).strip()
    return (s[:38] + "…") if len(s) > n else (s or "not present")
def acct_hover(a):   # full feature dump, each field truncated so the tooltip stays a tidy card
    r = _DFI.loc[a]
    return "<br>".join([f"<b>{a}</b>"] + [f"{c}: {_trunc(r[c])}" for c in _DFI.columns])

# ---- shared presentation config (editorial palette ported from viz.py) ----
import plotly.io as pio
INK="#22303C"; ACCENT="#D64545"; COOL="#3A7CA5"; WARN="#E8A33D"; GREEN="#4C956C"; GREY="#B7C0C7"; NAVY="#1F3A5F"
# muted, CVD-safe severity ramp (navy -> blue -> amber -> grey), red reserved for REVIEW/alerts
TIER_COLORS = {"CERTAIN":NAVY, "HIGH":COOL, "MEDIUM":WARN, "LOW":GREY, "REVIEW":ACCENT}
SIGNAL_TIER_COLORS = {"High":NAVY, "Medium":WARN, "Low":GREY}
# network edge colour by signal bucket (High signals navy, Medium amber)
HI_SIGS = {"email_alias","email","phone","device"}
SIG_COLORS = {s: (NAVY if s in HI_SIGS else WARN) for s in
              ["email_alias","email","phone","device","card","addr","name","ip_ua"]}
SINGLE_BAR = COOL
SIGLAB = {"ip_ua":"ip+useragent","addr":"address","email_alias":"email (disguise)"}
def siglab(s): return SIGLAB.get(s, s)
H_SHORT, H_STD, H_NET = 340, 430, 640
pio.templates["exec"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=12, color=INK),
    title=dict(x=0, font=dict(size=16)),
    margin=dict(l=70, r=30, t=60, b=55),
    xaxis=dict(automargin=True), yaxis=dict(automargin=True),
    uniformtext=dict(minsize=10, mode="hide"),
    legend=dict(bgcolor="rgba(255,255,255,0.6)"),
    hoverlabel=dict(bgcolor="white", bordercolor="#ccc", align="left", namelength=-1, font=dict(size=13))))
pio.templates.default = "plotly_white+exec"
print(f"{N} accounts, {df.shape[1]} columns")
df.head(3)

15008 accounts, 15 columns


,account_id,signup_ts,full_name,email,phone,addr_line1,addr_line2,city,state,zip,ip_address,payment_bin,card_last4,device_hash,user_agent
0,A000001,2026-03-25 00:13:26,Brandi Robinson,brandi_robinson@outlook.com,(577) 676-2256,9288 Kent Ave,,Port Daniel,LA,85318,104.29.154.211,453245,3188,d_13e0ad1624999c,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...
1,A000002,2026-03-17 03:29:09,Steven Rodriguez,stevenrodriguez75@aol.com,1-460-967-1187,5960 LONG AVE.,Apt 2A,New James,NJ,14717-2764,174.85.181.214,,,,Mozilla/5.0 (iPad; CPU OS 18_4 like Mac OS X) ...
2,A000003,2026-04-26 11:18:20,Lance Rogers,lancerogers@gmail.com,1-862-417-7104,9759 Wilson Ave,,Bowmanfort,PA,37459,205.254.166.183,542418,6898,d_3b69339ec7d222,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...


## 1. Data coverage — which signals can we even trust?

The deciding number is the **collision hub**: when a value is shared, by how many accounts?
**Small hub = trustworthy** (a device shared by 3 = one person). **Big hub = junk** (an IP shared by 46 = an office; a user-agent by 2,204 = "Chrome").

In [2]:
# STORY PLOT: fill rate — sparse fields can still be the strongest links (fill != trust)
fill = (df.drop(columns=["account_id"]).ne("").sum() / N * 100).sort_values()
fr = fill.reset_index(); fr.columns = ["field","fill_pct"]
fig = px.bar(fr, x="fill_pct", y="field", orientation="h", text=fr["fill_pct"].round(1),
             title="Coverage varies — but sparse fields can still be our strongest links")
fig.update_traces(texttemplate="%{text}%", textposition="outside", cliponaxis=False, marker_color=SINGLE_BAR,
                  hovertemplate="%{y}: %{x:.1f}% filled<extra></extra>")
fig.add_vline(x=80, line_dash="dash", line_color=ACCENT,
              annotation_text="fill-rate sparse cutoff = 80%", annotation_position="top")
fig.add_annotation(x=0, y=1.0, xref="paper", yref="paper", yanchor="bottom", xanchor="left", showarrow=False,
    font=dict(size=11, color="#777"), text="fill ≠ trust — device is only ~40% filled yet a top-tier signal")
fig.update_layout(height=H_SHORT, xaxis_title="% of accounts with a value", yaxis_title="", xaxis_range=[0,113])
fig

In [3]:
relay_nets = [ipaddress.ip_network(c) for c in pd.read_csv(INPUT/"icloud_relay_ranges.csv")["cidr"]]
def is_relay(ip):
    try: a = ipaddress.ip_address(ip)
    except ValueError: return False
    return any(a in n for n in relay_nets)
def phone10(s):
    d = re.sub(r"\D","",s); d = d[1:] if len(d)==11 and d[0]=="1" else d
    return d if len(d)==10 else None
def name_key(s):
    t = re.sub(r"[^\w\s]"," ",s.lower()).split(); return " ".join(sorted(t)) or None

_ip = df["ip_address"].where(~df["ip_address"].map(is_relay)).replace("", None)
raw = pd.DataFrame({
    "phone":      df["phone"].map(phone10),
    "card":       (df["payment_bin"]+"|"+df["card_last4"]).where(df["payment_bin"].ne("")&df["card_last4"].ne("")),
    "device":     df["device_hash"].replace("", None),
    "ip+useragent": (_ip+" || "+df["user_agent"]).where(_ip.notna()),
    "ip":         _ip,
    "user_agent": df["user_agent"].replace("", None),
    "name":       df["full_name"].map(name_key),
    "address":    (df["addr_line1"].str.upper().str.replace(r"[^\w\s]"," ",regex=True).str.split().str.join(" ")
                   +"|"+df["zip"].str.replace(r"\D","",regex=True).str[:5]).where(df["addr_line1"].ne("")),
})
def collision(col):
    vc = col.value_counts(); sh = vc[vc>=2]
    return pd.Series({"shared_vals":len(sh), "accts_shared":int(sh.sum()),
                      "max_hub":int(vc.max()) if len(vc) else 0,
                      "median_hub":float(sh.median()) if len(sh) else 0})
prof = pd.DataFrame({c: collision(raw[c]) for c in raw}).T
prof

,shared_vals,accts_shared,max_hub,median_hub
phone,2.0,7.0,4.0,3.5
card,482.0,1011.0,18.0,2.0
device,47.0,111.0,3.0,2.0
ip+useragent,40.0,215.0,11.0,5.0
ip,6.0,217.0,46.0,37.0
user_agent,7.0,15008.0,2204.0,2162.0
name,982.0,2278.0,12.0,2.0
address,56.0,190.0,18.0,2.0


In [4]:
# STORY PLOT: trust by collision hub — lollipop (a log axis has no bar baseline)
TIER = {"device":"High","phone":"High","card":"Medium","address":"Medium",
        "name":"Medium","ip+useragent":"Medium","ip":"Low","user_agent":"Low"}
HUBCAP_LBL = {"card":8,"address":12,"ip+useragent":10,"name":10}   # enforced ceilings from §5
p = prof.reset_index().rename(columns={"index":"signal"}); p["tier"] = p["signal"].map(TIER)
p = p.sort_values("max_hub")
fig = go.Figure()
for _, r in p.iterrows():
    fig.add_trace(go.Scatter(x=[1, r["max_hub"]], y=[r["signal"], r["signal"]], mode="lines",
        line=dict(color=GREY, width=2), hoverinfo="skip", showlegend=False))
for t, col in SIGNAL_TIER_COLORS.items():
    sub = p[p["tier"]==t]
    if len(sub):
        fig.add_trace(go.Scatter(x=sub["max_hub"], y=sub["signal"], mode="markers+text",
            text=sub["max_hub"], textposition="middle right", texttemplate="%{text:,}", cliponaxis=False,
            marker=dict(size=13, color=col), name=t,
            hovertemplate="%{y}: biggest shared group = %{x:,}<extra></extra>"))
for sg, cap in HUBCAP_LBL.items():
    fig.add_trace(go.Scatter(x=[cap], y=[sg], mode="markers",
        marker=dict(symbol="line-ns", size=18, line=dict(width=2, color="#555")),
        showlegend=False, hovertemplate=f"{sg}: suppressed above {cap}<extra></extra>"))
fig.add_vline(x=15, line_dash="dash", line_color=ACCENT,
              annotation_text="junk cutoff · hub>15", annotation_position="top")
fig.update_xaxes(type="log", title="biggest group sharing one value (log)",
                 range=[math.log10(0.8), math.log10(p["max_hub"].max()*4)])
fig.update_layout(title="Trust by collision hub size — small = trustworthy, big = junk",
    height=H_STD, yaxis_title="", legend_title="signal tier",
    annotations=[dict(x=0, y=1.09, xref="paper", yref="paper", showarrow=False, font=dict(size=11,color="#777"),
        text="This is the raw problem; the computed weights in §4 are the fix.")])
fig

## 2. Normalization — one identity key per field

Collapse each raw field to the value that identifies a *person*. Missing/untrusted → `None` (links nobody), but the **account always stays** (asserted below).
Email is special: same inbox reached via **different** dot/`+tag` spellings on a provider that ignores them = **disguise** = provably one person.

In [5]:
GMAIL = {"gmail.com","googlemail.com"}
ALIASING = GMAIL | {"outlook.com","hotmail.com","live.com","yahoo.com","proton.me","icloud.com","fastmail.com"}
def email_keys(s):
    e = s.strip().lower()
    if e.count("@")!=1: return None,None,False
    local,dom = e.split("@")
    if not local or not dom: return None,None,False
    if dom=="googlemail.com": dom="gmail.com"
    aliasable = dom in ALIASING
    clocal = local.split("+")[0]
    if dom in GMAIL: clocal = clocal.replace(".","")
    if not clocal: return None,None,False
    return f"{clocal}@{dom}", f"{local}@{dom}", aliasable

USPS = {"AVENUE":"AVE","AV":"AVE","BOULEVARD":"BLVD","STREET":"ST","STR":"ST","ROAD":"RD","LANE":"LN",
        "DRIVE":"DR","COURT":"CT","CIRCLE":"CIR","PLACE":"PL","HIGHWAY":"HWY","PARKWAY":"PKWY"}
DIRS = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W","NORTHEAST":"NE","NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"}
FAKE_PHONES = {"0000000000","1234567890","5555555555"}
def zip5(s):
    z = re.sub(r"\D","",s)[:5]; return z if len(z)==5 else None
def addr_key(l1,l2,z):
    z = zip5(z)
    toks = [USPS.get(DIRS.get(t,t),DIRS.get(t,t)) for t in re.sub(r"[^\w\s]"," ",l1.upper()).split()]
    street = " ".join(toks); unit = " ".join(re.sub(r"[^\w\s]"," ",l2.upper()).split())
    return f"{street}|{unit}|{z}" if street and z else None
def phone_ok(s):
    d = phone10(s)
    return None if (d is None or d[0] in "01" or d in FAKE_PHONES or len(set(d))==1) else d

ek = df["email"].map(email_keys)
em = pd.DataFrame(ek.tolist(), columns=["canon","raw","aliasable"])
K = pd.DataFrame({"account_id": df["account_id"]})
K["email"] = em["canon"]; K["email_raw"] = em["raw"]; K["email_aliasable"] = em["aliasable"]
K["phone"]  = df["phone"].map(phone_ok)
K["card"]   = (df["payment_bin"]+"|"+df["card_last4"]).where(df["payment_bin"].ne("")&df["card_last4"].ne(""))
K["device"] = df["device_hash"].replace("", None)
K["ip_ua"]  = (_ip+" || "+df["user_agent"]).where(_ip.notna())
K["addr"]   = [addr_key(a,b,z) for a,b,z in zip(df["addr_line1"],df["addr_line2"],df["zip"])]
K["zip5"]   = df["zip"].map(zip5)
K["name"]   = df["full_name"].map(name_key)
assert len(K)==N and K["account_id"].is_unique, "normalization lost accounts!"
print(f"normalized all {len(K)} accounts (none dropped). non-null keys:")
print(K.drop(columns=["account_id","email_raw","email_aliasable"]).notna().sum())

normalized all 15008 accounts (none dropped). non-null keys:
email     15008
phone     11740
card      10920
device     6017
ip_ua     14302
addr      15008
zip5      15008
name      15008
dtype: int64


In [6]:
# email disguise: same inbox, different spellings = one person, deliberately varied
g = em.dropna(subset=["canon"]).groupby("canon")
dis = g.agg(n=("raw","size"), n_raw=("raw","nunique"), aliasable=("aliasable","all"))
dis = dis[(dis.n>=2) & (dis.n_raw>1) & dis.aliasable]
print(f"disguise inboxes: {len(dis)}  covering {int(dis.n.sum())} accounts (all shared inboxes ARE disguise)")
# worked example: the busiest inbox
top = dis.sort_values("n", ascending=False).index[0]
df.loc[(em["canon"]==top).values, ["account_id","email","full_name"]].reset_index(drop=True)

disguise inboxes: 204  covering 473 accounts (all shared inboxes ARE disguise)


,account_id,email,full_name
0,A001308,jamiejacks.on@gmail.com,marcus webb
1,A002691,jamiejackson+new@gmail.com,marcus webb
2,A002938,jamiejacks.on+promo@gmail.com,marcus webb
3,A007585,jamiejackson@gmail.com,"Webb, Marcus"
4,A007784,ja.miejackson+shop@gmail.com,Marcus X. Webb
5,A008340,jamiejackson+52@gmail.com,"Webb, Marcus"
6,A009291,jamiejackso.n+shop@gmail.com,Marcus L. Webb
7,A011275,jamiejackson+offers@gmail.com,Marcus M. Webb
8,A012022,jamie.jackson@gmail.com,M. Webb


In [7]:
# CHART: disguise ring sizes — the ground-truth same-actor groups the weights learn from
ring = dis["n"].value_counts().sort_index().reset_index()
ring.columns = ["accounts_in_ring","num_inboxes"]; ring["total_accounts"] = ring.accounts_in_ring*ring.num_inboxes
fig = px.bar(ring, x="accounts_in_ring", y="num_inboxes", text="num_inboxes", custom_data=["total_accounts"],
             title=f"Disguise rings = the ground truth — {len(dis):,} inboxes hide {int(dis['n'].sum()):,} accounts",
             labels={"accounts_in_ring":"accounts sharing ONE hidden inbox","num_inboxes":"# inboxes"})
fig.update_traces(marker_color=SINGLE_BAR, textposition="outside", cliponaxis=False,
    hovertemplate="%{x} accounts/inbox<br>%{y:,} inboxes<br>%{customdata[0]:,} accounts total<extra></extra>")
fig.add_annotation(x=0, y=1.0, xref="paper", yref="paper", yanchor="bottom", xanchor="left", showarrow=False,
    font=dict(size=11, color="#777"), text="these ≥2-account rings are the m-numerator the §4 weights learn from")
fig.update_layout(height=H_SHORT, xaxis=dict(type="category"),
                  yaxis=dict(range=[0, ring.num_inboxes.max()*1.18]))
fig

## 3. Address fuzzy — catch typos, refuse apartment merges

Exact keys miss typos (`ANDEESON` vs `ANDERSON`). We fuzzy-match **within a ZIP block**, but only accept it when house-number and unit are identical and just the street *name* differs — so `APT 18` never merges with `APT 11`. Address stays corroborate-only (it never links a pair alone).

In [8]:
from difflib import SequenceMatcher
FUZZY = 0.90
def unit_tokens(s): return tuple(t for t in s.split() if t.isdigit() or t in {"APT","UNIT","STE","#"})
street = K["addr"].dropna().str.split("|").str[0]
byz = K.loc[street.index].assign(street=street).groupby("zip5")
exact, fuzzy_ok, fuzzy_bad = 0, [], 0
for z, grp in byz:
    if len(grp)<2: continue
    for (i,a),(j,b) in combinations(grp.iterrows(), 2):
        if a["street"]==b["street"]: exact += 1
        elif SequenceMatcher(None,a["street"],b["street"]).ratio() >= FUZZY:
            if unit_tokens(a["street"])==unit_tokens(b["street"]): fuzzy_ok.append((a["account_id"],b["account_id"]))
            else: fuzzy_bad += 1
print(f"address pairs: exact {exact} | fuzzy kept (real typos) {len(fuzzy_ok)} | fuzzy dropped (apt merges) {fuzzy_bad}")

class UF:
    def __init__(self): self.p={}
    def find(self,x):
        self.p.setdefault(x,x)
        while self.p[x]!=x: self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        ra,rb=self.find(a),self.find(b)
        if ra!=rb: self.p[max(ra,rb)]=min(ra,rb)
ufa = UF()
for _, grp in K.dropna(subset=["addr"]).groupby("addr"):
    ids = grp["account_id"].tolist()
    for o in ids[1:]: ufa.union(ids[0], o)
for a,b in fuzzy_ok: ufa.union(a,b)
root = {a: ufa.find(a) for a in K.loc[K["addr"].notna(),"account_id"]}
gs = pd.Series(list(root.values())).value_counts()
K["addr_group"] = K["account_id"].map(lambda a: root.get(a) if a in root and gs[root[a]]>=2 else None)
print(f"address groups (>=2 accts): {K['addr_group'].nunique()}")

address pairs: exact 933 | fuzzy kept (real typos) 259 | fuzzy dropped (apt merges) 84


address groups (>=2 accts): 62


In [9]:
# CHART: address matching — typos kept, apartment merges refused (protects legit customers)
amt = pd.DataFrame({"kind":["exact match","fuzzy KEPT (real typo)","false merges prevented"],
                    "pairs":[exact, len(fuzzy_ok), fuzzy_bad]})
fig = px.bar(amt, x="pairs", y="kind", orientation="h", text="pairs")
fig.update_traces(marker_color=[GREY, COOL, GREEN], texttemplate="%{text:,}", textposition="outside",
                  cliponaxis=False, hovertemplate="%{y}: %{x:,} pairs<extra></extra>")
fig.add_annotation(x=0, y=1.0, xref="paper", yref="paper", yanchor="bottom", xanchor="left", showarrow=False,
    font=dict(size=11, color="#777"),
    text="fuzzy rule: accept ≥ 0.90 similarity AND identical unit tokens → APT 18 never merges with APT 11")
fig.update_layout(title="Address matching keeps typos, refuses apartment merges (protects legit customers)",
                  height=H_SHORT, showlegend=False, xaxis_title="account pairs", yaxis_title="",
                  xaxis_range=[0, amt.pairs.max()*1.18])
fig

## 4. Computed weights — `BASE = log2(m/u)`

We don't hand-pick weights — the data sets them. For each signal we ask two plain questions:

**m — "do the *same* people reuse it?"**
Look only at account pairs we *know* are one person (the disguise rings). How often do they share this field?
> *Card example:* of same-person pairs that both added a card, **17%** used the exact same card → **m = 0.17**.

**u — "do *strangers* share it by accident?"**
Now take two random accounts. What's the chance they share this field by pure luck?
> *Card example:* almost never — about **0.01%** → **u = 0.0001**.

**BASE = log2( m ÷ u )** — how many times more the *same* people share it than strangers do.
> *Card:* 0.17 vs 0.0001 = **~1,600× more** → a high, trustworthy weight.

Plain version: a field that only the same person reuses (and strangers rarely hit) earns a **big** weight; a field everyone shares earns almost none.

We sort signals into three trust tiers (same as the coverage chart):
- **High** — `email`, `device`, `phone`: identity fields, **link a pair on their own**.
- **Medium** — `card`, `address`, `name`, `ip+useragent`: corroborate, **need a second Medium** (or a High) to link.
- **Low** — `ip` alone, `user_agent` alone: never link (only their combination ip+useragent is used, and it is Medium).

This tiering is the guard: computed `name` has the highest weight (people reuse their name), but name is **Medium**, so a lone rare "John Smith" can never link strangers. Weights rank confidence; tiers decide what may link.

In [10]:
# labels: pairs inside a disguise inbox = provably same actor
labels = []
email_members = defaultdict(list)
for a, c in zip(K["account_id"], K["email"]):
    if pd.notna(c): email_members[c].append(a)
Kx = K.set_index("account_id")
for c, mem in email_members.items():
    if len(mem)>=2:
        sub = Kx.loc[mem]
        if sub["email_aliasable"].all() and sub["email_raw"].nunique()>1:
            labels += list(combinations(mem, 2))
print(f"labeled same-actor pairs: {len(labels)}")

def m_of(col):
    v = Kx[col]; both=hit=0
    for a,b in labels:
        x,y = v.loc[a], v.loc[b]
        if pd.notna(x) and pd.notna(y): both+=1; hit += (x==y)
    return hit/both if both else 0.0
def u_of(col):
    vc = K[col].dropna().value_counts(); n=int(vc.sum()); return float(((vc/n)**2).sum())

BASE, mu = {}, []
for s in ["email","card","device","phone","addr","name","ip_ua"]:
    m,u = m_of(s), u_of(s)
    BASE[s] = math.log2(m/u) if m>0 else 0.0
    mu.append({"signal":s, "m":round(m,3), "u":f"{u:.2e}", "BASE=log2(m/u)":round(BASE[s],2)})
BASE["email_alias"] = BASE["email"]      # disguise = same inbox
comp = pd.DataFrame(mu).sort_values("BASE=log2(m/u)", ascending=False)
comp

labeled same-actor pairs: 387


,signal,m,u,BASE=log2(m/u)
0,email,1.000,7.01e-05,13.80
5,name,0.230,8.28e-05,11.44
1,card,0.167,1.04e-04,10.65
3,phone,0.044,8.53e-05,9.02
2,device,0.058,1.71e-04,8.40
4,addr,0.018,6.74e-05,8.07
6,ip_ua,0.000,7.55e-05,0.00


In [11]:
# STORY PLOT: computed BASE weights, tier-colored (High links alone, Medium needs 2)
TIERMAP = {"email":"High","device":"High","phone":"High",
           "card":"Medium","addr":"Medium","name":"Medium","ip_ua":"Medium"}
b = comp.copy(); b["BASE"] = b["BASE=log2(m/u)"]; b["tier"] = b["signal"].map(TIERMAP)
b["label"] = b["signal"].map(siglab); b = b.sort_values("BASE")
fig = px.bar(b, x="BASE", y="label", orientation="h", color="tier", text="BASE",
             category_orders={"tier":["High","Medium"]}, color_discrete_map=SIGNAL_TIER_COLORS,
             title="Computed weight  BASE = log2(m/u)  — High links alone, Medium needs a second signal")
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}: weight %{x:.2f}<extra></extra>")
fig.add_annotation(x=b.loc[b.signal=="name","BASE"].iloc[0], y="name",
    text="highest weight — but capped Medium, so a lone 'John Smith' never links strangers",
    showarrow=True, arrowhead=2, ax=-30, ay=-45, bgcolor="white", font=dict(size=10,color="#555"), align="right")
fig.add_annotation(x=0, y=1.0, xref="paper", yref="paper", yanchor="bottom", xanchor="left", showarrow=False,
    font=dict(size=11, color="#555"),
    text="tier gate — blue (High: email/device/phone) links a pair ALONE; orange (Medium) needs ≥2")
fig.add_vline(x=0, line_color="#bbb", annotation_text="0 = unused", annotation_position="bottom left")
fig.update_layout(height=H_STD, xaxis_title="computed weight", yaxis_title="",
                  xaxis_range=[0, b.BASE.max()*1.2], legend_title="signal tier")
fig

## 5. Weighted edges → clusters

Every shared key drops `BASE × hub_penalty` onto each pair that shares it; weights **add up**.
`hub_penalty` shrinks a value shared by many (infrastructure); email is exempt (a private inbox is one person however many accounts use it).
**Tier gate — keep a pair as an edge if it has a High signal (links alone) OR at least two Medium signals (corroboration).** Weights then rank confidence. Connected edges = a cluster.

In [12]:
SOFT   = {"email_alias":8,"email":4,"card":3,"device":3,"phone":3,"addr":4,"name":3,"ip_ua":4}
HUBCAP = {"card":8,"addr":12,"ip_ua":10,"name":10}      # drop values shared by more (collision/building)
# trust tiers: High links alone, Medium needs >=2, Low unused
CERTAINTY = {"email_alias"}
HIGH      = {"email","device","phone"}
MEDIUM    = {"card","addr","name","ip_ua"}
LINKS_ALONE = CERTAINTY | HIGH
NO_PEN = {"email_alias","email"}
def hub_penalty(dfc, sig): return 1.0 if sig in NO_PEN else 1.0/(1.0+max(0,dfc-SOFT[sig]))

index = defaultdict(lambda: defaultdict(list))
for sig,col in [("phone","phone"),("card","card"),("device","device"),("ip_ua","ip_ua"),("addr","addr_group"),("name","name")]:
    for a,v in zip(K["account_id"], K[col]):
        if v is not None and not (isinstance(v,float) and pd.isna(v)): index[sig][v].append(a)

pair_w = defaultdict(float); pair_sig = defaultdict(list); pair_types = defaultdict(set)
def add(sig, members):
    dfc = len(members)
    if dfc<2 or (sig in HUBCAP and dfc>HUBCAP[sig]) or BASE.get(sig,0)<=0: return
    w = BASE[sig]*hub_penalty(dfc, sig)
    for a,b in combinations(sorted(members),2):
        pair_w[(a,b)] += w; pair_sig[(a,b)].append((sig,round(w,2))); pair_types[(a,b)].add(sig)
for c,mem in email_members.items():
    if len(mem)<2: continue
    sub = Kx.loc[mem]
    add("email_alias" if (sub["email_aliasable"].all() and sub["email_raw"].nunique()>1) else "email", mem)
for sig,idx in index.items():
    for _,mem in idx.items(): add(sig, mem)

uf = UF(); kept=[]
for (a,b),w in sorted(pair_w.items()):
    t = pair_types[(a,b)]
    if (t & LINKS_ALONE) or (len(t & MEDIUM) >= 2):     # tier gate
        uf.union(a,b); kept.append((a,b,round(w,2)))
comp_c = defaultdict(list)
for a in K["account_id"]:
    if a in uf.p: comp_c[uf.find(a)].append(a)
clusters = {r:sorted(m) for r,m in comp_c.items() if len(m)>1}
linked = sum(len(m) for m in clusters.values())
print(f"edges kept {len(kept)} | clusters {len(clusters)} | accounts linked {linked}")

edges kept 470 | clusters 251 | accounts linked 587


In [13]:
# tier + rank, then write deliverables
edge_sigs = defaultdict(set)
for a,b,w in kept:
    for s,_ in pair_sig[(a,b)]: edge_sigs[uf.find(a)].add(s)
colmap = {"email_alias":"email","email":"email","phone":"phone","card":"card","device":"device","ip_ua":"ip_ua","addr":"addr","name":"name"}
def sval(mem,col):
    vc = Kx.loc[mem,col].dropna().value_counts(); return vc.index[0] if len(vc) else None
rows=[]
for r,mem in clusters.items():
    sigs=edge_sigs[r]; n=len(mem); nt=len(sigs)
    alias="email_alias" in sigs; high=bool(sigs&HIGH); nmed=len(sigs&MEDIUM)
    tier = "CERTAIN" if alias else "HIGH" if high else "MEDIUM" if nmed>=2 else "LOW"
    if n>25 and not alias: tier="REVIEW"
    mx=max(w for a,b,w in kept if uf.find(a)==r)
    links="; ".join(f"{s}={sval(mem,colmap[s])}" for s in sorted(sigs) if sval(mem,colmap[s]))
    rows.append({"members":mem,"size":n,"tier":tier,"confidence":round(mx*(1+0.4*(nt-1)),2),
                 "dominant":max(sigs,key=lambda s:BASE[s]),"links":links})
order={"REVIEW":5,"CERTAIN":4,"HIGH":3,"MEDIUM":2,"LOW":1}
rows.sort(key=lambda c:(order[c["tier"]],c["confidence"],c["size"]),reverse=True)
for i,c in enumerate(rows,1): c["cid"]=f"C{i:05d}"

with open(OUT/"clusters.csv","w",newline="") as f:
    wr=csv.writer(f); wr.writerow(["account_id","cluster_id","cluster_size","confidence_tier","dominant_signal","linking_values"])
    for c in rows:
        for a in c["members"]: wr.writerow([a,c["cid"],c["size"],c["tier"],c["dominant"],c["links"]])
with open(OUT/"edges.csv","w",newline="") as f:
    wr=csv.writer(f); wr.writerow(["account_a","account_b","combined_weight","signals"])
    for a,b,w in sorted(kept): wr.writerow([a,b,w,"|".join(f"{s}:{sw}" for s,sw in pair_sig[(a,b)])])
tiers=Counter(c["tier"] for c in rows)
with open(OUT/"findings.md","w") as f:
    f.write(f"# Ranked findings (computed-base)\n\n{N} accounts | {len(rows)} clusters | {linked} linked\nTiers: {dict(tiers)}\n\n")
    for c in rows[:15]:
        f.write(f"### {c['cid']} — {c['size']} accts — {c['tier']} (conf {c['confidence']})\n- Linked by: {c['links']}\n- Members: {', '.join(c['members'][:12])}{' …' if c['size']>12 else ''}\n\n")
print("tiers:", dict(tiers), "-> wrote clusters.csv, edges.csv, findings.md")

tiers: {'CERTAIN': 204, 'HIGH': 44, 'MEDIUM': 3} -> wrote clusters.csv, edges.csv, findings.md


In [14]:
# KPI cards — headline numbers (viz.py-style go.Indicator row)
from plotly.subplots import make_subplots
cards = [(N,"accounts"), (linked,"linked"), (len(rows),"clusters"), (tiers.get("CERTAIN",0),"CERTAIN rings")]
kpi = make_subplots(rows=1, cols=len(cards), specs=[[{"type":"indicator"}]*len(cards)])
for i,(val,lab) in enumerate(cards,1):
    kpi.add_trace(go.Indicator(mode="number", value=val,
        number=dict(font=dict(size=44, color=NAVY), valueformat=","),
        title=dict(text=lab, font=dict(size=13, color=INK))), row=1, col=i)
kpi.update_layout(height=200, margin=dict(l=20,r=20,t=30,b=10))
kpi

In [15]:
# STORY PLOTS: tier mix (donut, severity-ordered, KPI in hole) + cluster-size spread
torder = ["CERTAIN","HIGH","MEDIUM","LOW","REVIEW"]
present = [(t, tiers[t]) for t in torder if tiers.get(t,0) > 0]
labs = [t for t,_ in present]; vals = [v for _,v in present]
pulls = [0.12 if t=="REVIEW" else 0 for t in labs]
ch = sum(tiers.get(t,0) for t in ("CERTAIN","HIGH"))
tcol = ["black" if t in ("LOW","MEDIUM") else "white" for t in labs]
donut_ann = [dict(text=f"<b>{linked:,}</b><br>accounts<br>linked", x=0.5, y=0.5, showarrow=False, font=dict(size=22, color=NAVY))]
if "REVIEW" in labs:
    donut_ann.append(dict(text="REVIEW = size>25 & not email-alias", x=0.5, y=-0.08,
                          xref="paper", yref="paper", showarrow=False, font=dict(size=10, color="#777")))
f1 = go.Figure(go.Pie(labels=labs, values=vals, hole=0.58, sort=False, pull=pulls,
    marker=dict(colors=[TIER_COLORS[t] for t in labs]),
    textinfo="label+value", textposition="inside", insidetextorientation="horizontal",
    textfont=dict(color=tcol, size=13),
    hovertemplate="%{label}: %{value} clusters (%{percent})<extra></extra>"))
f1.update_layout(height=H_STD, title=f"{ch} of {len(rows)} clusters are CERTAIN/HIGH — investigate first",
    annotations=donut_ann)
display(f1)
sizes = pd.Series([c["size"] for c in rows])
sc = sizes.value_counts()
f2 = px.histogram(sizes, title="Most actors run 2 accounts — the long tail is the organized rings")
f2.update_traces(xbins=dict(start=1.5, size=1), marker_color=SINGLE_BAR,
                 hovertemplate="cluster size %{x}<br>%{y:,} clusters<extra></extra>")
f2.add_vline(x=25.5, line_dash="dash", line_color=ACCENT,
             annotation_text="size>25 → REVIEW", annotation_position="top")
f2.update_layout(height=H_STD, showlegend=False, xaxis_title="accounts in cluster",
                 yaxis_title="clusters (log)", yaxis_type="log",
                 yaxis_range=[math.log10(0.7), math.log10(sc.max()*1.4)])
f2

## 6. Worked example — one cluster, end to end

Trace the cluster containing `A004729` (the Cabrera case): raw fields → keys → which collide → weight per pair → why it forms.

In [16]:
anchor = "A004729"
if anchor in uf.p:
    r = uf.find(anchor); mem = clusters[r]
    print(f"cluster of {anchor}: {mem}\n")
    print(Kx.loc[mem, ["email","phone","card","device","addr"]].to_string())
    print("\nkept edges (pair -> signals):")
    for a,b,w in kept:
        if uf.find(a)==r:
            print(f"  {a}-{b}: " + " + ".join(f"{s}({sw})" for s,sw in pair_sig[(a,b)]) + f" = {w} >= TAU")
    c = next(c for c in rows if anchor in c["members"])
    print(f"\n=> {c['cid']} | size {c['size']} | {c['tier']} | headline {c['dominant']} | conf {c['confidence']}")

cluster of A004729: ['A000047', 'A004729', 'A006216']

                               email       phone         card            device                    addr
account_id                                                                                             
A000047     tammycabrera78@gmail.com  4899927688          NaN  d_9cfdc4a98f2cae  803 ANDEESON DR||15627
A004729     rogercabrera30@gmail.com  3395115722  440066|3240  d_9cfdc4a98f2cae  803 ANDERSON DR||15627
A006216         mcabrera@outlook.com  6807274656          NaN  d_9cfdc4a98f2cae  803 ANDERSON DR||15627

kept edges (pair -> signals):
  A000047-A004729: device(8.4) + addr(8.07) = 16.47 >= TAU
  A000047-A006216: device(8.4) + addr(8.07) = 16.47 >= TAU
  A004729-A006216: device(8.4) + addr(8.07) = 16.47 >= TAU

=> C00205 | size 3 | HIGH | headline device | conf 23.06


In [17]:
# CHART: this cluster as a network (nodes=accounts colored by tier; hover edge for shared signals)
import math as _m
if anchor in uf.p:
    r = uf.find(anchor); mem = clusters[r]; n = len(mem)
    cc = next(c for c in rows if anchor in c["members"])
    pos = {a:(_m.cos(2*_m.pi*i/n), _m.sin(2*_m.pi*i/n)) for i,a in enumerate(mem)}
    fig = go.Figure()
    seen = set()
    for a,b,w in kept:
        if uf.find(a)==r:
            sigs = [s for s,_ in pair_sig[(a,b)]]
            hi = any(s in HI_SIGS for s in sigs)
            col = NAVY if hi else WARN; bucket = "High-signal link" if hi else "Medium-signal link"
            lbl = " + ".join(siglab(s) for s in sigs)
            fig.add_trace(go.Scatter(x=[pos[a][0],pos[b][0]], y=[pos[a][1],pos[b][1]], mode="lines",
                line=dict(color=col,width=2), hovertext=lbl, hoverinfo="text",
                name=bucket, legendgroup=bucket, showlegend=bucket not in seen))
            seen.add(bucket)
    fig.add_trace(go.Scatter(x=[pos[a][0] for a in mem], y=[pos[a][1] for a in mem], mode="markers+text",
        text=mem, textposition="top center", textfont=dict(size=11),
        marker=dict(size=18, color=ACCENT, line=dict(color="white",width=2)),
        hovertext=[acct_hover(a) for a in mem], hoverinfo="text", showlegend=False))
    fig.update_layout(title=f"{cc['cid']} — {cc['tier']} — linked by: {cc['links']}", height=H_STD,
        legend_title="cluster tier",
        xaxis=dict(visible=False, range=[-1.7,1.7]), yaxis=dict(visible=False, range=[-1.6,1.6], scaleanchor="x"))
    display(fig)

## 7. The funnel, in numbers (live)

Interactive funnel of how 15,008 accounts narrow to the linked set — every number pulled from this run.

In [18]:
# funnel stays in ACCOUNTS (candidate PAIRS are a different unit -> kept in the print, not the bars)
stages = {"raw accounts": N, "normalized (kept)": len(K), "accounts linked": linked}
pct = 100*linked/N
fig = go.Figure(go.Funnel(y=list(stages.keys()), x=list(stages.values()),
    textinfo="value+percent initial", texttemplate="%{value:,}  (%{percentInitial})",
    hovertemplate="%{y}: %{x:,} accounts (%{percentInitial} of raw)<extra></extra>",
    marker={"color":[NAVY, COOL, WARN]}))
fig.update_layout(height=H_STD,
    title=f"{N:,} signups → {linked:,} linked accounts ({pct:.1f}%) across {len(rows):,} actors")
display(fig)
print("FUNNEL:", stages, "| candidate pairs:", len(pair_w), "| edges:", len(kept),
      "| singletons dropped:", N-linked, "| tiers:", dict(tiers))

FUNNEL: {'raw accounts': 15008, 'normalized (kept)': 15008, 'accounts linked': 587} | candidate pairs: 3032 | edges: 470 | singletons dropped: 14421 | tiers: {'CERTAIN': 204, 'HIGH': 44, 'MEDIUM': 3}


## 8. Explore clusters — account network

Each cluster as a small graph: **nodes = accounts**, **edges = shared signals** that link them. Use the dropdown to step through every cluster one by one (ranked C00001 = highest confidence). Hover a node for its account id and name.

In [19]:
import math as _m
edges_by_root = defaultdict(list)
for a,b,w in kept: edges_by_root[uf.find(a)].append((a,b,w))
ordered = sorted([c for c in rows if c["size"] >= 3], key=lambda c: (-c["size"], c["cid"]))   # explorable (>=3), biggest first

# each edge coloured by WHICH signal links the pair (dominant = strongest signal on that edge)
SIG_ORDER = ["email_alias","email","device","phone","card","addr","name","ip_ua"]
SIG_EDGE = {"email_alias":"#1F3A5F","email":"#3A7CA5","device":"#4C956C","phone":"#2E8B8B",
            "card":"#E8A33D","addr":"#9467BD","name":"#8C564B","ip_ua":"#B7C0C7"}
def dom_sig(sigs): return min(sigs, key=lambda s: SIG_ORDER.index(s))

fig = go.Figure()
cluster_ranges = []                              # (start, end) trace-index span per cluster
for c in ordered:
    start = len(fig.data)
    mem = c["members"]; n = len(mem); root = uf.find(mem[0])
    pos = {a:(_m.cos(2*_m.pi*i/n), _m.sin(2*_m.pi*i/n)) for i,a in enumerate(mem)}
    buckets = {}                                 # signal -> [xs, ys]
    for a,b,w in edges_by_root[root]:
        sg = dom_sig([s for s,_ in pair_sig[(a,b)]])
        buckets.setdefault(sg, [[],[]])
        buckets[sg][0] += [pos[a][0],pos[b][0],None]; buckets[sg][1] += [pos[a][1],pos[b][1],None]
    for sg,(ex,ey) in buckets.items():
        fig.add_trace(go.Scatter(x=ex,y=ey,mode="lines",line=dict(color=SIG_EDGE[sg],width=2.2),
            name=siglab(sg), hovertemplate=f"linked by <b>{siglab(sg)}</b><extra></extra>",
            visible=False, showlegend=False))
    fig.add_trace(go.Scatter(x=[pos[a][0] for a in mem],y=[pos[a][1] for a in mem],
        mode="markers+text", text=mem, textposition="top center", textfont=dict(size=11),
        marker=dict(size=16,color=ACCENT,line=dict(color="white",width=2)),
        hovertext=[acct_hover(a) for a in mem], hoverinfo="text", visible=False, showlegend=False))
    cluster_ranges.append((start, len(fig.data)))
total = len(fig.data)
for i in range(*cluster_ranges[0]): fig.data[i].visible = True
for sg in SIG_ORDER:                             # static legend: edge colour -> linking signal
    fig.add_trace(go.Scatter(x=[None],y=[None],mode="lines",line=dict(color=SIG_EDGE[sg],width=3),
        name=siglab(sg), showlegend=True))

n_leg = len(SIG_ORDER); total_all = total + n_leg      # cluster traces + static legend traces
buttons = []
for k,c in enumerate(ordered):
    s,e = cluster_ranges[k]
    vis = [False]*total_all
    for i in range(s,e): vis[i]=True                   # this cluster's edges+nodes
    for i in range(total, total_all): vis[i]=True      # legend ALWAYS visible (plotly cycles short arrays)
    buttons.append(dict(label=f"{c['size']:>2} accts · {c['cid']} · {c['tier']}", method="update",
        args=[{"visible":vis},
              {"title":f"{c['cid']} — {c['size']} accounts — {c['tier']} — linked by: {c['links']}"}]))
first = ordered[0]
fig.update_layout(
    updatemenus=[dict(buttons=buttons, x=0, y=1.02, yanchor="bottom", xanchor="left", showactive=True, pad=dict(r=10,t=4))],
    title=dict(text=f"{first['cid']} — {first['size']} accounts — {first['tier']} — linked by: {first['links']}",
               y=0.98, yanchor="top"),
    height=H_NET, legend_title="edge colour = linked by", margin=dict(t=130),
    xaxis=dict(visible=False, range=[-1.6,1.6]), yaxis=dict(visible=False, range=[-1.4,1.4], scaleanchor="x"))
fig.add_annotation(x=0.5, y=-0.04, xref="paper", yref="paper", showarrow=False, font=dict(size=11, color="#777"),
    text=f"{len(ordered)} clusters with ≥3 accounts shown · 2-account pairs (one link each) omitted — all in clusters.csv")
print(f"network dropdown: {len(ordered)} explorable clusters (>=3 accts), biggest first")
fig

network dropdown: 58 explorable clusters (>=3 accts), biggest first
